# Meme Hunter MFE 

**Architecture:**
- Target Labeling: Maximum Forward Excursion (MFE) / Horizon 12h / Min Pump 10%
- LightGBM Classifier (Binary)
- Walk-Forward Expanding Window Filter + 15 Day Purge Gap
- Hit & Run 10% Features: Micro-Volume, Price Acceleration, Order Flow Proxy + Market Context
- Output: Probability of +10% pump over next 12h

In [2]:
import os, sys, glob, gc
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CẤU HÌNH HỆ THỐNG
# ==========================================
TIMEFRAME = '1h'
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

WORKING_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('./ml/training/models')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path('/kaggle/input/datasets/hungbui317/macd-coin/macd-overlay - Copy/data') if IS_KAGGLE else Path('./data')
OHLCV_DIR = DATA_DIR / 'ohlcv'
OI_DIR = Path('/kaggle/input/datasets/hungbui317/macd-coin/data/dat') 
OUTPUT_FILE = str(WORKING_DIR / f'features_{TIMEFRAME}_full.parquet')

TF_CONFIG = {
    '1h':  {'rule': '1h',  'min_bars': 300, 'unit': '1h bars'},
}
CFG = TF_CONFIG[TIMEFRAME]

print(f"🚀 Khởi động Pipeline | Output: {OUTPUT_FILE}")

# ==========================================
# 2. HÀM ĐỌC & CHUẨN HÓA DỮ LIỆU THÔ
# ==========================================
def load_ohlcv_1h(symbol):
    for name in [f"{symbol}_USDT.parquet", f"{symbol}.parquet"]:
        fp = OHLCV_DIR / name
        if fp.exists():
            df = pd.read_parquet(fp)
            if 'timestamp' not in df.columns and 'open_time' in df.columns: 
                df = df.rename(columns={'open_time':'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms') if df['timestamp'].dtype=='int64' else pd.to_datetime(df['timestamp'])
            return df.sort_values('timestamp').reset_index(drop=True)
    return pd.DataFrame()

def load_oi_1h(symbol):
    """
    Hàm load OI thông minh: Thử mọi biến thể tên file có thể có.
    """
    # Các biến thể tên file có thể tồn tại trong folder
    possible_names = [f"{symbol}USDT.parquet", ]
    
    for name in possible_names:
        fp = OI_DIR / name
        if fp.exists():
            try:
                df_oi = pd.read_parquet(fp)
                if 'timestamp' in df_oi.columns:
                    df_oi['timestamp'] = pd.to_datetime(df_oi['timestamp'])
                
                # Ép kiểu để tiết kiệm RAM
                float_cols = ['top_ls_ratio', 'global_ls_ratio', 'oi_change_1h', 'oi_change_24h']
                for col in float_cols:
                    if col in df_oi.columns:
                        df_oi[col] = df_oi[col].astype('float32')
                return df_oi.sort_values('timestamp').reset_index(drop=True)
            except:
                print(fp)
                continue
    return pd.DataFrame()

def resample_1h(df_1h, tf):
    if df_1h.empty: return pd.DataFrame()
    return df_1h.set_index('timestamp').resample(TF_CONFIG.get(tf, {'rule':tf})['rule']).agg({
        'open':'first', 'high':'max', 'low':'min', 'close':'last', 'volume':'sum'
    }).dropna().reset_index()

print("✓ Đã nạp Cell 1 (Hệ thống & Đọc file).")

🚀 Khởi động Pipeline | Output: /kaggle/working/features_1h_full.parquet
✓ Đã nạp Cell 1 (Hệ thống & Đọc file).


In [3]:
# ==========================================
# 3. CÁC HÀM TÍNH TOÁN KỸ THUẬT (PHỤC HỒI NGUYÊN BẢN)
# ==========================================
def calculate_rsi(prices, period=14):
    d = prices.diff(); g = d.where(d>0,0).rolling(period).mean(); l = (-d.where(d<0,0)).rolling(period).mean()
    return 100-(100/(1+g/(l.replace(0,np.nan)+1e-9)))

def calculate_macd(df, fast=12, slow=26, signal=9):
    ef = df['close'].ewm(span=fast).mean()
    es = df['close'].ewm(span=slow).mean()
    # NÂNG CHUẨN: Chia cho close để có giá trị tương đối (%)
    df['macd'] = (ef - es) / df['close'] 
    df['macd_signal'] = df['macd'].ewm(span=signal).mean()
    df['macd_histogram'] = df['macd'] - df['macd_signal']
    df['macd_slope'] = df['macd'].diff()
    return df
def add_time_since_features(df):
    # Tính số nến kể từ lần Sweep gần nhất
    for col in ['bullish_sweep', 'bearish_sweep']:
        # Tạo mask: 1 nếu có sweep, 0 nếu không
        mask = df[col] == 1
        # Groupby các cụm 0 để đếm số nến liên tiếp không có sweep
        df[f'bars_since_{col}'] = mask.groupby((mask != mask.shift()).cumsum()).cumcount()
        # Chuẩn hóa để tránh số quá lớn (ví dụ: chia cho 100 nến)
        df[f'bars_since_{col}'] = df[f'bars_since_{col}'].clip(0, 500) / 100.0
    return df
def calculate_features_v2(df):
    df = df.copy()
    # 1. Cơ bản
    df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
    df['atr_14'] = tr.rolling(14).mean() # Giả định tr đã tính ở ngoài hoặc trong hàm
    
    # 2. CHUẨN HÓA CÁC KHOẢNG CÁCH (Nâng chuẩn)
    # Thay vì dùng (Close - EMA), ta dùng (Close - EMA) / ATR để biết nó "quá xa" bao nhiêu lần biến động
    for e in [21, 50, 200]:
        df[f'dist_to_ema_{e}_zscore'] = (df['close'] - df[f'ema_{e}']) / (df['atr_14'] + 1e-9)

    # 3. VOLATILITY COMPRESSION (Trái tim của Tam Giác)
    bb_mid = df['close'].rolling(20).mean()
    bb_std = df['close'].rolling(20).std()
    kc_range = df['atr_14'] * 1.5
    # Tỷ lệ nén: BB Width / KC Width. < 1 nghĩa là đang nén cực chặt
    df['squeeze_ratio'] = (bb_std * 2) / (kc_range + 1e-9)
    
    # 4. MOMENTUM SỨC MẠNH (Dùng cho Phân loại)
    df['rsi_14'] = calculate_rsi(df['close'], 14)
    df['rsi_norm'] = df['rsi_14'] / 100.0 # Ép về 0-1 cho máy học dễ nuốt
    
    return df

def calculate_liquidity_sweep(df, lookback=20):
    df = df.copy()
    df['swing_low'] = df['low'].rolling(window=lookback).min().shift(1)
    df['swing_high'] = df['high'].rolling(window=lookback).max().shift(1)
    df['candle_range'] = df['high'] - df['low'] + 1e-9
    df['lower_wick'] = df[['open', 'close']].min(axis=1) - df['low']
    df['upper_wick'] = df['high'] - df[['open', 'close']].max(axis=1)
    df['lower_wick_ratio'] = df['lower_wick'] / df['candle_range']
    df['upper_wick_ratio'] = df['upper_wick'] / df['candle_range']
    df['vol_sma_20'] = df['volume'].rolling(20).mean()
    
    cond_sweep_bottom = df['low'] < df['swing_low']
    cond_reject_bottom = df['close'] > df['swing_low']
    cond_pinbar_bottom = df['lower_wick_ratio'] > 0.3
    cond_vol_surge = df['volume'] > df['vol_sma_20']
    df['bullish_sweep'] = (cond_sweep_bottom & cond_reject_bottom & cond_pinbar_bottom & cond_vol_surge).astype(int)
    
    cond_sweep_top = df['high'] > df['swing_high']
    cond_reject_top = df['close'] < df['swing_high']
    cond_pinbar_top = df['upper_wick_ratio'] > 0.3
    df['bearish_sweep'] = (cond_sweep_top & cond_reject_top & cond_pinbar_top & cond_vol_surge).astype(int)
    
    # Ghi đè tín hiệu sweep vào tín hiệu cross gốc của bạn
    df['macd_cross_up'] = df['bullish_sweep']
    df['macd_cross_down'] = df['bearish_sweep']
    return df

def calculate_features(df):
    df=df.copy(); df['log_returns']=np.log(df['close']/df['close'].shift(1))
    df['high_low_range']=(df['high']-df['low'])/df['close']; df['body_size']=abs(df['close']-df['open'])/df['close']
    df['candle_range']=df['high']-df['low']+1e-9; df['lower_wick']=df[['open','close']].min(axis=1)-df['low']; df['upper_wick']=df['high']-df[['open','close']].max(axis=1)
    df['lower_wick_ratio_current']=df['lower_wick']/df['candle_range']; df['upper_wick_ratio_current']=df['upper_wick']/df['candle_range']
    
    for p in [7,14,21,50,200]: df[f'ema_{p}']=df['close'].ewm(span=p).mean()
    for p in [10,20,50,200]: df[f'sma_{p}']=df['close'].rolling(p).mean()
    
    tr=pd.concat([df['high']-df['low'],abs(df['high']-df['close'].shift(1)),abs(df['low']-df['close'].shift(1))],axis=1).max(axis=1)
    df['atr_14']=tr.rolling(14).mean(); df['volatility_14']=df['log_returns'].rolling(14).std()
    df['vol_sma_14']=df['volatility_14'].rolling(14).mean(); df['vol_compression']=df['volatility_14']/(df['vol_sma_14']+1e-9)
    df['volume_sma_20']=df['volume'].rolling(20).mean(); df['volume_std_20']=df['volume'].rolling(20).std()
    df['volume_ratio']=df['volume']/(df['volume_sma_20']+1e-9); df['volume_zscore']=(df['volume']-df['volume_sma_20'])/(df['volume_std_20']+1e-9)
    df['volume_trend']=df['volume'].rolling(7).mean()/(df['volume'].rolling(21).mean()+1e-9); df['volume_spike']=(df['volume_ratio']>2).astype(int)
    
    df['rsi_14']=calculate_rsi(df['close'], 14); df['rsi_slope']=df['rsi_14'].diff(3)
    l14=df['low'].rolling(14).min(); h14=df['high'].rolling(14).max()
    df['stoch_k']=100*(df['close']-l14)/(h14-l14).replace(0,np.nan); df['stoch_d']=df['stoch_k'].rolling(3).mean()
    df['roc_7']=df['close'].pct_change(7); df['roc_14']=df['close'].pct_change(14)
    
    # Phase 11 Features
    df['sma_30']=df['close'].rolling(30).mean(); df['price_vs_sma_30']=df['close']/(df['sma_30']+1e-9)
    df['momentum_30']=df['close'].pct_change(30)
    pdm=df['high'].diff(); mdm=-df['low'].diff()
    pdm=pdm.where((pdm>mdm)&(pdm>0),0); mdm=mdm.where((mdm>pdm)&(mdm>0),0); atr_s=tr.rolling(14).mean()
    pdi=100*(pdm.rolling(14).mean()/atr_s.replace(0,np.nan)); mdi=100*(mdm.rolling(14).mean()/atr_s.replace(0,np.nan))
    df['adx']=(100*abs(pdi-mdi)/(pdi+mdi).replace(0,np.nan)).rolling(14).mean()
    df['dist_to_high_30d']=(df['close']-df['high'].rolling(30).max())/df['close']
    df['dist_to_low_30d']=(df['close']-df['low'].rolling(30).min())/df['close']
    for e in [21,50,200]: df[f'dist_to_ema_{e}_pct']=(df['close']-df[f'ema_{e}'])/df['close']
    
    df['trend_state']=np.where(df['close']>df['sma_50'],1,np.where(df['close']<df['sma_50'],-1,0))
    df['is_trending']=(df['adx']>25).astype(int); df['is_volatile']=(df['vol_compression']>1.5).astype(int)
    df['hour_sin']=np.sin(2*np.pi*df['timestamp'].dt.hour/24); df['hour_cos']=np.cos(2*np.pi*df['timestamp'].dt.hour/24)
    df['day_sin']=np.sin(2*np.pi*df['timestamp'].dt.dayofweek/7); df['day_cos']=np.cos(2*np.pi*df['timestamp'].dt.dayofweek/7)
    df['vol_ratio_alpha']=df['volume_ratio']*df['volatility_14']
    df['market_structure_bull']=((df['close']>df['sma_200'])&(df['sma_50']>df['sma_200'])).astype(int)
    
    bb_mid=df['close'].rolling(20).mean(); bb_std=df['close'].rolling(20).std()
    bb_wd=(bb_mid+2*bb_std - (bb_mid-2*bb_std))/(bb_mid+1e-9)
    df['bb_squeeze']=(bb_wd<bb_wd.rolling(20).quantile(0.2)).astype(int)
    df['vwap_30d']=(df['close']*df['volume']).rolling(30).sum()/(df['volume'].rolling(30).sum()+1e-9)
    df['above_poc']=(df['close']>df['vwap_30d']).astype(int)
    
    # Hit & Run 10% Features
    df['micro_volume']=df['volume']/(df['volume'].rolling(5).mean()+1e-9)
    df['price_accel']=df['close'].pct_change(1)/(df['close'].pct_change(4).replace(0,np.nan)+1e-9)
    df['order_flow_proxy']=(df['close']-df['low'])/(df['high']-df['low']+1e-9)
    
    df=calculate_macd(df); df=df.drop(columns=['macd_cross_up','macd_cross_down'], errors='ignore')
    df=calculate_liquidity_sweep(df)
    df['usd_vol_24h'] = (df['volume'] * df['close']).rolling(24).sum()
    

    # --- NORMALIZE ABSOLUTE FEATURES ---
    df['atr_14_relative'] = df['atr_14'] / (df['close'] + 1e-9)
    df['vwap_30d_dist'] = (df['close'] - df['vwap_30d']) / (df['vwap_30d'] + 1e-9)
    df['usd_vol_24h_log'] = np.log1p(df['usd_vol_24h'] + 1e-9)
    df['swing_low_dist'] = (df['close'] - df['swing_low']) / (df['atr_14'] + 1e-9)
    df['swing_high_dist'] = (df['close'] - df['swing_high']) / (df['atr_14'] + 1e-9)
    
    return df.dropna(subset=['macd','swing_low','vol_sma_20','vwap_30d','usd_vol_24h'])

# ==========================================
# 4. HÀM TÍNH TOÁN DÒNG TIỀN (OPEN INTEREST)
# ==========================================

    
def merge_and_engineer_oi(df_price, symbol):
    df_oi = load_oi_1h(symbol)
    oi_base_cols = ['top_ls_ratio', 'global_ls_ratio', 'oi_change_1h', 'oi_change_24h']
    
    if df_oi.empty:
        for col in oi_base_cols: df_price[col] = np.nan
        df_price['has_oi'] = 0
        return df_price

    # --- DEBUG SECTION START ---
    # Chỉ debug cho con OG hoặc con đầu tiên để không làm rác log
    if symbol == '0G' or symbol == '0GUSDT':
        print(f"\n[DEBUG OI] Symbol: {symbol}")
        print(f" - Price Timestamp Range: {df_price['timestamp'].min()} -> {df_price['timestamp'].max()}")
        print(f" - OI Timestamp Range:    {df_oi['timestamp'].min()} -> {df_oi['timestamp'].max()}")
        print(f" - Price Timezone: {df_price['timestamp'].dt.tz} | OI Timezone: {df_oi['timestamp'].dt.tz}")
        
        # Kiểm tra xem có nến nào khớp giờ không
        p_set = set(df_price['timestamp'].dt.floor('h'))
        o_set = set(df_oi['timestamp'].dt.floor('h'))
        intersection = p_set.intersection(o_set)
        print(f" - Số lượng nến khớp giờ (Intersection): {len(intersection)}")
        if len(intersection) == 0:
            print(" ⚠️ CẢNH BÁO: Không có nến nào khớp giờ! Kiểm tra lại Timezone hoặc định dạng nến.")
    # --- DEBUG SECTION END ---

    # Chuẩn hóa Timestamp (Ép về không múi giờ để merge)
    df_price['timestamp'] = pd.to_datetime(df_price['timestamp']).dt.tz_localize(None)
    df_oi['timestamp'] = pd.to_datetime(df_oi['timestamp']).dt.tz_localize(None)

    df_price = df_price.sort_values('timestamp')
    df_oi = df_oi.sort_values('timestamp')
    
    df = pd.merge_asof(
        df_price, 
        df_oi[['timestamp'] + oi_base_cols], 
        on='timestamp', 
        direction='backward',
        tolerance=pd.Timedelta('1h') 
    )
    
    df['has_oi'] = df['oi_change_1h'].notna().astype(int)
    
    # Tính toán Features
    mask = df['has_oi'] == 1
    df.loc[mask, 'smart_money_div'] = df['top_ls_ratio'] - df['global_ls_ratio']
    df.loc[mask, 'oi_price_interact_1h'] = df['close'].pct_change(1).fillna(0) * df['oi_change_1h']
    
    return df    

# ==========================================
# 5. XỬ LÝ NHÃN, SHIFT & TỐI ƯU RAM
# ==========================================
def generate_momentum_labels(df, horizon=12, min_pump=0.10):
    df_rev = df.iloc[::-1].copy()
    future_max_high = df_rev['high'].rolling(horizon, min_periods=1).max()
    df['future_max_high'] = future_max_high.sort_index()
    df['max_pump_pct'] = (df['future_max_high'] - df['close']) / df['close']
    df['label'] = (df['max_pump_pct'] >= min_pump).astype(int)
    
    if 'usd_vol_24h' in df.columns: df.loc[df['usd_vol_24h'] < 1000000, 'label'] = np.nan
    return df.drop(columns=['future_max_high'])

def apply_feature_shift(df):
    ex = {'timestamp','symbol','open','high','low','close','volume','label','macd_cross_up', 'macd_cross_down'}
    sc = [c for c in df.columns if c not in ex]
    df[sc] = df[sc].shift(1) 
    return df.dropna(subset=sc[:3])
    
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if not pd.api.types.is_numeric_dtype(col_type): continue
        c_min, c_max = df[col].min(), df[col].max()
        if pd.api.types.is_integer_dtype(col_type):
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)
    return df

print("✓ Đã nạp Cell 2 (Feature Engineering BẢN FULL).")

✓ Đã nạp Cell 2 (Feature Engineering BẢN FULL).


In [ ]:
symbols = [f.stem.replace('_USDT','') for f in OHLCV_DIR.glob('*.parquet')]
symbols = [s for s in symbols if not any(x in s for x in ['-26','-25','-24'])]
print(f"🔍 Tìm thấy {len(symbols)} symbols. Đang khởi tạo Bối cảnh BTC...")

# 1. TẠO BỐI CẢNH BTC TRƯỚC (Sửa lỗi chưa định nghĩa biến)
btc_context = pd.DataFrame()
btc_sym = 'BTCUSDT' if 'BTCUSDT' in symbols else ('BTC' if 'BTC' in symbols else None)
if btc_sym:
    btc_df = calculate_features(resample_1h(load_ohlcv_1h(btc_sym), TIMEFRAME))
    btc_context = btc_df[['timestamp','close','sma_200','log_returns']].copy()
    btc_context.columns = ['timestamp','btc_close','btc_sma_200','btc_returns']
    btc_context['btc_is_bull'] = (btc_context['btc_close'] > btc_context['btc_sma_200']).astype(int)

# 2. VÒNG LẶP CHÍNH
print("\n🚀 BẮT ĐẦU CHẠY PIPELINE TỔNG HỢP...")
all_data = []

for sym in symbols:
    try:
        # Load & Cơ bản
        d1 = load_ohlcv_1h(sym)
        if d1.empty: continue
        dt = resample_1h(d1, TIMEFRAME)
        if len(dt) < CFG['min_bars']: continue
        
        dt['symbol'] = sym
        dt = calculate_features(dt)
        
        # Bối cảnh 1D
        d1d = resample_1h(d1, '1d') 
        d1d['ema_200_1d'] = d1d['close'].ewm(span=200).mean()
        d1d['rsi_14_1d'] = calculate_rsi(d1d['close'], 14)
        
        d1d_feat = d1d[['timestamp', 'ema_200_1d', 'rsi_14_1d']].copy()
        d1d_feat['timestamp'] = d1d_feat['timestamp'] + pd.Timedelta(days=1) 
        
        dt = dt.sort_values('timestamp')
        d1d_feat = d1d_feat.sort_values('timestamp')
        dt = pd.merge_asof(dt, d1d_feat, on='timestamp', direction='backward')
        
        for c in ['ema_200_1d', 'rsi_14_1d']: dt[c] = dt[c].ffill().fillna(50 if 'rsi' in c else 0)
        dt['ema_200_1d_dist'] = (dt['close'] - dt['ema_200_1d']) / dt['close']

        # Dòng tiền OI
        # dt = merge_and_engineer_oi(dt, sym)
        
        # Merge bối cảnh BTC
        if not btc_context.empty:
            dt = dt.merge(btc_context, on='timestamp', how='left')
            for c in ['btc_is_bull', 'btc_returns']: dt[c] = dt[c].ffill().fillna(0)
            dt['rs_vs_btc'] = dt['log_returns'] - dt['btc_returns']
            
        # Shift, Ép RAM & Gán Nhãn
        dt = apply_feature_shift(dt)
        if dt.empty: continue
        dt = reduce_mem_usage(dt)
        dt = generate_momentum_labels(dt, horizon=12, min_pump=0.10) 
        
        # Xóa các dòng rác cuối dataframe do label future sinh ra
        dt = dt.dropna(subset=['label'])
        
        all_data.append(dt)
        print(f"  ✓ {sym}: {len(dt)} bars")
        
        # Xả RAM theo Batch
        if len(all_data) >= 50 or sym == symbols[-1]:
            df_batch = pd.concat(all_data, ignore_index=True)
            df_batch.to_parquet(WORKING_DIR / f'batch_features_{sym}.parquet', index=False)
            del all_data, df_batch
            all_data = [] 
            gc.collect()
            
    except Exception as e: 
        print(f"  ✗ Lỗi tại {sym}: {e}")

# 3. CHỐT SỔ VÀ LƯU FILE CUỐI CÙNG
print("\n✓ Đang gom các Batch lại...")
batch_files = glob.glob(str(WORKING_DIR / 'batch_features_*.parquet'))
df_final = pd.read_parquet(batch_files)

for f in batch_files: os.remove(f) # Dọn dẹp

df_final.to_parquet(OUTPUT_FILE, index=False)
print(f"✅ HOÀN TẤT! File lưu tại: {OUTPUT_FILE} ({len(df_final)} rows)")

y = df_final['label'].values
print(f"📊 Thống kê Nhãn Target (>10% Pump): Tổng {len(y)} nến | Tỷ lệ nổ: {y.mean()*100:.2f}%")

In [4]:
df_final=pd.read_parquet(OUTPUT_FILE)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import precision_score, recall_score, average_precision_score

print("🪓 Đang chia tách dữ liệu theo Trục Thời Gian...")
# df_final = pd.read_parquet(OUTPUT_FILE)

# Chốt chặn 1: Sort tuyệt đối theo thời gian
df_final = df_final.sort_values('timestamp').reset_index(drop=True)

# Định nghĩa Features (Lọc bỏ các cột string/datetime hoặc target)
EXCLUDE_COLS = [
    # 1. Các chỉ báo trùng lặp (Correlation > 0.95)
    'volume_sma_20', 'sma_10', 'ema_7', 'ema_21', 'sma_20', 'sma_30', 'ema_14',
    'sma_50', 'sma_200', 'ema_100',
    
    # 2. Các biến rác đang bị NaN (Bạn cần kiểm tra lại file OI sau)
    'smart_money_div', 'ls_momentum_4h', 'oi_price_interact_1h', 'oi_zscore_24h', 'is_oi_squeeze',
    
    # 3. Các cột rò rỉ hoặc phi định lượng đã biết
        'atr_14', 'vwap_30d', 'usd_vol_24h', 'swing_low', 'swing_high', 'vol_sma_20', 'candle_range', 'lower_wick', 'upper_wick',
    'timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'label', 
    'max_pump_pct', 'future_return', 'future_max_high', 'trade_result', 'ignition', 'btc_close',''
]
FEATURES = [c for c in df_final.columns if c not in EXCLUDE_COLS]

# Chốt chặn 2: Chia Train / Val / Test (70 - 15 - 15)
n_samples = len(df_final)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

train_df = df_final.iloc[:train_end]
val_df = df_final.iloc[train_end:val_end]
test_df = df_final.iloc[val_end:]

X_train, y_train = train_df[FEATURES], train_df['label']
X_val, y_val = val_df[FEATURES], val_df['label']
X_test, y_test = test_df[FEATURES], test_df['label']

print(f"✓ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"✓ Feature count: {len(FEATURES)}")

In [5]:
print("🔍 ĐANG KIỂM TRA TÌNH TRẠNG DỮ LIỆU OI TRÊN DF_FINAL...")

# 1. Kiểm tra tỷ lệ lấp đầy (Fill rate)
oi_cols = [
    'top_ls_ratio', 'global_ls_ratio', 'oi_change_1h', 'oi_change_24h',
    'smart_money_div', 'ls_momentum_4h', 'oi_price_interact_1h', 'oi_zscore_24h', 'is_oi_squeeze'
]

# Kiểm tra xem các cột có tồn tại không
existing_oi_cols = [c for c in oi_cols if c in df_final.columns]
print(f"Các cột OI tìm thấy trong DF: {existing_oi_cols}")

if not existing_oi_cols:
    print("❌ THẤT BẠI: Không tìm thấy bất kỳ cột OI nào trong df_final!")
else:
    # Tính tỷ lệ % dữ liệu có giá trị (không phải NaN và không phải 0)
    audit_data = []
    for col in existing_oi_cols:
        total = len(df_final)
        nan_count = df_final[col].isna().sum()
        zero_count = (df_final[col] == 0).sum()
        valid_pct = ((total - nan_count - zero_count) / total) * 100
        audit_data.append({'Feature': col, 'Valid_Data_%': f"{valid_pct:.2f}%", 'NaNs': nan_count, 'Zeros': zero_count})
    
    audit_df = pd.DataFrame(audit_data)
    print("\n📊 BẢNG THỐNG KÊ ĐỘ PHỦ OI:")
    print(audit_df.to_string(index=False))

    # 2. Kiểm tra sâu: OI có tập trung ở vài symbol hay rải đều?
    print("\n🕵️ KIỂM TRA PHÂN BỔ THEO SYMBOL:")
    # Lấy 1 cột đại diện là 'oi_change_1h'
    if 'oi_change_1h' in df_final.columns:
        oi_by_sym = df_final.groupby('symbol')['oi_change_1h'].apply(lambda x: (x != 0).sum() > 0)
        symbols_with_oi = oi_by_sym[oi_by_sym == True].index.tolist()
        print(f"  - Tổng số Symbol có dữ liệu OI: {len(symbols_with_oi)} / {df_final['symbol'].nunique()}")
        if len(symbols_with_oi) < 10:
            print(f"  - Danh sách symbol có OI: {symbols_with_oi}")
            print("  ⚠️ CẢNH BÁO: Data OI quá mỏng, model sẽ không thể học được pattern chung!")

    # 3. Kiểm tra logic tương quan nhanh
    if 'oi_change_1h' in df_final.columns and 'label' in df_final.columns:
        sample_corr = df_final[['oi_change_1h', 'label']].corr().iloc[0,1]
        print(f"\n🔗 Tương quan nhanh OI_Change vs Label: {sample_corr:.4f}")

🔍 ĐANG KIỂM TRA TÌNH TRẠNG DỮ LIỆU OI TRÊN DF_FINAL...
Các cột OI tìm thấy trong DF: ['top_ls_ratio', 'global_ls_ratio', 'oi_change_1h', 'oi_change_24h']

📊 BẢNG THỐNG KÊ ĐỘ PHỦ OI:
        Feature Valid_Data_%    NaNs  Zeros
   top_ls_ratio        0.00% 9636625      0
global_ls_ratio        0.00% 9636625      0
   oi_change_1h        0.00% 9636625      0
  oi_change_24h        0.00% 9636625      0

🕵️ KIỂM TRA PHÂN BỔ THEO SYMBOL:
  - Tổng số Symbol có dữ liệu OI: 572 / 572

🔗 Tương quan nhanh OI_Change vs Label: nan


In [ ]:
print("\n🤖 Đang huấn luyện LightGBM Sniper...")

# Thiết lập siêu tham số tối ưu cho dữ liệu nhiễu tài chính
params = {
    'objective': 'binary',
    'metric': 'average_precision', # PR-AUC: Cực kỳ quan trọng cho imbalanced data
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'max_depth': 5,             # Cây nông để chống học vẹt (overfit)
    'num_leaves': 24,
    'feature_fraction': 0.7,    # Lấy random 70% feature mỗi cây để tạo tính đa dạng
    'class_weight': 'balanced', # Ép model chú ý vào các lệnh pump hiếm hoi
    'n_jobs': -1,
    'random_state': 42,
    'verbose': -1
}

# Tạo Dataset của LightGBM
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

# Fit model tích hợp Early Stopping
evals_result = {}
model = lgb.train(
    params,
    dtrain,
    num_boost_round=1500,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.record_evaluation(evals_result)
    ]
)

print("✅ Huấn luyện hoàn tất!")

In [ ]:
import matplotlib.pyplot as plt

print("\n🎯 KIỂM ĐỊNH TRÊN TẬP THỰC CHIẾN (OUT-OF-SAMPLE TEST)")

# Dự đoán xác suất trên tập Test
y_pred_proba = model.predict(X_test)

# Đánh giá PR-AUC (Kỳ vọng > 0.3 là xuất sắc trong trading)
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"🏆 PR-AUC Score: {pr_auc:.4f}")

# Tìm ngưỡng bóp cò (Threshold) tối ưu hóa Precision
# AI tự tin > 99% mới bắn, bỏ qua các kèo 50/50
threshold = np.percentile(y_pred_proba, 99) 
y_pred_bin = (y_pred_proba >= threshold).astype(int)

precision = precision_score(y_test, y_pred_bin, zero_division=0)
recall = recall_score(y_test, y_pred_bin, zero_division=0)

print("-" * 40)
print(f"Ngưỡng bóp cò (Threshold Top 15%): {threshold:.4f}")
print(f"Tổng số kèo báo: {y_pred_bin.sum()} / {len(y_test)}")
print(f"Tỷ lệ Thắng thực tế (Precision): {precision*100:.2f}%")
print(f"Tỷ lệ Bắt trúng sóng (Recall): {recall*100:.2f}%")
print("-" * 40)

# Mổ xẻ não bộ AI: Nó đang dùng cái gì để ra quyết định?
importance = pd.DataFrame({
    'feature': FEATURES,
    'gain': model.feature_importance(importance_type='gain')
}).sort_values('gain', ascending=False)

print("\n🔍 TOP 10 FEATURES QUYẾT ĐỊNH CÚ PUMP:")
print(importance.head(10).to_string(index=False))

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Đảm bảo đường dẫn chuẩn với môi trường của bạn (sửa lại nếu cần)
WORKING_DIR = Path('/kaggle/working') 

print("📊 TRẠM 1: ĐANG KHÁM SỨC KHỎE DỮ LIỆU TỔNG THỂ...")

try:
    df = df_final
    print(f"✓ Đã load Data: {df.shape[0]} dòng, {df.shape[1]} cột.")
except Exception as e:
    raise FileNotFoundError(f"❌ Không tìm thấy file {OUTPUT_FILE}. Chạy lại bước trước! Lỗi: {e}")

# 1. Quét Rác (NaN & Infinite)
# Lỗi chia 0 (1e-9) đôi khi vẫn tạo ra số quá lớn (Inf) làm AI chết sặc
df = df.replace([np.inf, -np.inf], np.nan)
initial_len = len(df)
# df = df.dropna()
print(f"🧹 Đã dọn dẹp {initial_len - len(df)} dòng chứa NaN/Inf (Giữ lại {len(df)} dòng sạch).")

# 2. Phân tích Phân phối Cú Pump (Max Pump Pct)
if 'max_pump_pct' in df.columns:
    pump_stats = df['max_pump_pct'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    print("\n📈 THỐNG KÊ BIÊN ĐỘ BẬT TĂNG (TRONG 12 NẾN TƯƠNG LAI):")
    print(f"  - Trung vị (Median): {pump_stats['50%']*100:.2f}% (Đây là mức mà 50% số kèo đạt được)")
    print(f"  - Top 10% kèo mạnh nhất (90th pct): {pump_stats['90%']*100:.2f}%")
    print(f"  - Top 1% kèo điên rồ nhất (99th pct): {pump_stats['99%']*100:.2f}%")
    
    # Tỷ lệ Win Rate tự nhiên với mốc 10%
    win_rate = (df['max_pump_pct'] >= 0.10).mean() * 100
    print(f"\n🎯 TỶ LỆ NỔ KÈO TỰ NHIÊN (>10%): {win_rate:.2f}%")
    if win_rate < 5:
        print("  ⚠️ BÁO ĐỘNG: Tỷ lệ nổ quá thấp! Khuyên dùng mốc Target động theo ATR thay vì hardcode 10%.")
else:
    print("❌ Không tìm thấy cột 'max_pump_pct'.")



📊 TRẠM 1: ĐANG KHÁM SỨC KHỎE DỮ LIỆU TỔNG THỂ...
✓ Đã load Data: 9636625 dòng, 94 cột.
🧹 Đã dọn dẹp 0 dòng chứa NaN/Inf (Giữ lại 9636625 dòng sạch).

📈 THỐNG KÊ BIÊN ĐỘ BẬT TĂNG (TRONG 12 NẾN TƯƠNG LAI):
  - Trung vị (Median): 2.11% (Đây là mức mà 50% số kèo đạt được)
  - Top 10% kèo mạnh nhất (90th pct): 6.91%
  - Top 1% kèo điên rồ nhất (99th pct): 19.54%

🎯 TỶ LỆ NỔ KÈO TỰ NHIÊN (>10%): 4.73%
  ⚠️ BÁO ĐỘNG: Tỷ lệ nổ quá thấp! Khuyên dùng mốc Target động theo ATR thay vì hardcode 10%.


In [8]:
print("\n🔬 TRẠM 2: ĐO LƯỜNG SỨC MẠNH DỰ BÁO CỦA TỪNG TÍNH NĂNG (IC - Information Coefficient)...")

# Loại bỏ các cột phi định lượng hoặc rò rỉ tương lai
cols_to_drop = [
    'timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'label', 
    'ignition', 'trade_result', 'max_pump_pct', 'future_return', 'top_ls_ratio',
    'atr_14', 'vwap_30d', 'usd_vol_24h', 'swing_low', 'swing_high', 'vol_sma_20', 
    'candle_range', 'lower_wick', 'upper_wick'
]
features = [c for c in df.columns if c not in cols_to_drop]

# Tính Spearman Correlation (Đo lường mối quan hệ phi tuyến, tốt hơn Pearson trong trading)
# So sánh Feature với Nhãn (label) và Biên độ thực tế (max_pump_pct)
correlations = []
for f in features:
    try:
        # Tương quan với tỷ lệ pump thực tế
        corr_val = df[f].corr(df['max_pump_pct'], method='spearman')
        correlations.append({'Feature': f, 'IC': corr_val, 'Abs_IC': abs(corr_val)})
    except:
        pass

corr_df = pd.DataFrame(correlations).sort_values('Abs_IC', ascending=False)

print("🏆 TOP 15 TÍNH NĂNG DẪN DẮT THỊ TRƯỜNG (Có tương quan mạnh nhất với cú Pump):")
print(corr_df[['Feature', 'IC']].head(15).to_string(index=False))

print("\n🗑️ TOP 5 TÍNH NĂNG VÔ DỤNG NHẤT (Nên cân nhắc loại bỏ để giảm nhiễu):")
print(corr_df[['Feature', 'IC']].tail(5).to_string(index=False))


🔬 TRẠM 2: ĐO LƯỜNG SỨC MẠNH DỰ BÁO CỦA TỪNG TÍNH NĂNG (IC - Information Coefficient)...
🏆 TOP 15 TÍNH NĂNG DẪN DẮT THỊ TRƯỜNG (Có tương quan mạnh nhất với cú Pump):
         Feature        IC
      vol_sma_14  0.388705
   volatility_14  0.387526
  high_low_range  0.371224
 vol_ratio_alpha  0.320809
dist_to_high_30d -0.237072
 dist_to_low_30d  0.236701
       body_size  0.230804
   volume_sma_20  0.144950
      vol_sma_20  0.144950
   volume_std_20  0.144207
     usd_vol_24h  0.125602
      ema_200_1d -0.062061
    volume_ratio  0.061713
         day_sin  0.061699
             adx  0.060004

🗑️ TOP 5 TÍNH NĂNG VÔ DỤNG NHẤT (Nên cân nhắc loại bỏ để giảm nhiễu):
        Feature  IC
   top_ls_ratio NaN
global_ls_ratio NaN
   oi_change_1h NaN
  oi_change_24h NaN
         has_oi NaN


In [ ]:
print("\n🕵️ TRẠM 3: TRUY QUÉT TÍN HIỆU TRÙNG LẶP (ĐA CỘNG TUYẾN)...")

# Chỉ xét Top 30 features mạnh nhất để tiết kiệm thời gian tính toán
top_features = corr_df['Feature'].head(30).tolist()
df_top = df[top_features]

# Tính ma trận tương quan giữa các Feature với nhau
feature_corr = df_top.corr(method='spearman').abs()

# Lọc ra các cặp có độ tương quan > 0.85 (Chỉ báo trùng lặp)
redundant_pairs = []
for i in range(len(feature_corr.columns)):
    for j in range(i+1, len(feature_corr.columns)):
        if feature_corr.iloc[i, j] > 0.85:
            redundant_pairs.append({
                'Feature 1': feature_corr.columns[i],
                'Feature 2': feature_corr.columns[j],
                'Correlation': feature_corr.iloc[i, j]
            })

redundant_df = pd.DataFrame(redundant_pairs).sort_values('Correlation', ascending=False)

if not redundant_df.empty:
    print("⚠️ BÁO ĐỘNG: BẠN ĐANG CÓ CÁC CHỈ BÁO KÝ SINH LẪN NHAU (>85% giống nhau)!")
    print("   Quy tắc tối ưu: Giữ lại cột có IC (ở Trạm 2) cao hơn, vứt cột kia đi.")
    print(redundant_df.head(10).to_string(index=False))
else:
    print("✅ Tuyệt vời! Các tính năng Top đầu của bạn hoàn toàn độc lập, không bị đa cộng tuyến.")

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

def train_cascade_sniper(golden_df):
    print("\n🤖 Tầng 2: Đang huấn luyện AI Sniper (Chế độ Baseline MFE)...")
    
    # 1. Chốt chặn an toàn: Đảm bảo data đã sort theo thời gian để test Out-Of-Sample chuẩn
    golden_df = golden_df.sort_values('timestamp').reset_index(drop=True)
    
    # 2. Chia Train/Test (80% Quá khứ / 20% Hiện tại)
    split_idx = int(len(golden_df) * 0.8)
    train_df = golden_df.iloc[:split_idx]
    test_df = golden_df.iloc[split_idx:]
    
    X_tr = train_df[FEATURES]
    y_tr = train_df['label']
    X_te = test_df[FEATURES]
    y_te = test_df['label']
    
    # Ép kiểu dữ liệu về float để LightGBM không bị lỗi "Cannot cast object"
    X_tr = X_tr.apply(pd.to_numeric, errors='coerce')
    X_te = X_te.apply(pd.to_numeric, errors='coerce')
    
    # 3. Khởi tạo LightGBM (Giữ nguyên setup cân bằng class của bạn)
    clf = lgb.LGBMClassifier(
        n_estimators=300, 
        learning_rate=0.05, 
        max_depth=4,             # Cây nông để AI tập trung vào Core Pattern, tránh học vẹt
        class_weight='balanced', # Bắt buộc vì class 1 chỉ chiếm ~15%
        random_state=42, 
        n_jobs=-1,
        verbose=-1
    )
    
    # 4. Fit Model
    clf.fit(X_tr, y_tr)
    
    # 5. Đánh giá trên tập Test
    preds_proba = clf.predict_proba(X_te)[:, 1]
    
    # Tính Precision ở ngưỡng tự tin cao (Top 20% kèo AI chắc chắn nhất)
    threshold = np.percentile(preds_proba, 80)
    bot_calls = preds_proba >= threshold
    
    true_wins = y_te[bot_calls].sum()
    total_calls = bot_calls.sum()
    precision = true_wins / total_calls if total_calls > 0 else 0
    
    # 6. In Báo cáo Thực chiến
    print(f"\n{'='*40}\n KẾT QUẢ THỰC CHIẾN (OOS TEST)\n{'='*40}")
    print(f"Tổng số kèo Hợp lưu trong tập Test: {len(y_te)}")
    print(f"Tỷ lệ nổ Tự nhiên (Nếu đánh mù mờ): {y_te.mean()*100:.2f}%")
    print("-" * 40)
    print(f"Ngưỡng bóp cò (Threshold Top 20%): {threshold:.4f}")
    print(f"AI đã bóp cò: {total_calls} kèo")
    print(f"Số kèo Win (>10%): {true_wins}")
    print(f"🏆 AI PRECISION (Tỷ lệ Thắng thực tế): {precision*100:.2f}%")
    
    # 7. Xem AI đang "nhìn" vào đâu để quyết định
    importance = pd.DataFrame({'feature': FEATURES, 'gain': clf.feature_importances_}).sort_values('gain', ascending=False)
    print("\n🔍 Top 5 Features quan trọng nhất định đoạt cú Pump:")
    print(importance.head(5).to_string(index=False))
    
    return clf, threshold

# Bóp cò:
lightgbm_model, best_threshold = train_cascade_sniper(df_final)

In [ ]:
import zipfile
import os
from IPython.display import FileLink

# Nén tệp parquet thành zip
with zipfile.ZipFile('features.zip', 'w') as zipf:
    zipf.write('/kaggle/working/features_1h_full.parquet', arcname='features_1h_full.parquet')

# Tạo link tải tệp zip
FileLink('features.zip')
